# lb-whisper: fine-tune Whisper on Luxembourgish (Colab / Kaggle)

Runtime → Change runtime type → **T4 GPU** (or better). Everything is driven by the scripts in the repo; this notebook only sequences them.

In [ ]:
!nvidia-smi -L
!git clone https://github.com/Sitraka17/lb-whisper.git
%cd lb-whisper/finetune
!pip install -q -r requirements.txt
!apt-get -qq install -y ffmpeg > /dev/null

## Hugging Face login
The token needs **write** access only if you push the model at the end. Common Voice no longer comes from the Hub (see section 1).

In [ ]:
from huggingface_hub import login
login()

## Keep checkpoints on Drive (Colab only)
Skip on Kaggle; use `/kaggle/working` instead.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/lb-whisper/runs
!ln -sfn /content/drive/MyDrive/lb-whisper/runs runs

## 1. Data
Since October 2025 Common Voice is distributed only through **Mozilla Data Collective**. Download the Luxembourgish (`lb`) archive there (free account, accept the terms), put it in `MyDrive/lb-whisper/cv-lb.tar.gz`, then run the cell. The ZLS set is **not** used for the published weights (conflicting licences, see the README).

In [ ]:
!mkdir -p data/cv_raw && tar -xzf /content/drive/MyDrive/lb-whisper/cv-lb.tar.gz -C data/cv_raw
!find data/cv_raw -maxdepth 3 -name 'train.tsv'
# point --dir at the folder that contains clips/ and train.tsv
!python prepare_data.py --source cv-dir --cv-dir "$(dirname $(find data/cv_raw -maxdepth 3 -name train.tsv | head -1))" --out data/cv_lb
!cat data/cv_lb/stats.json

## 2. Baseline (zero-shot): measure before training

In [ ]:
!python evaluate.py --model openai/whisper-large-v3-turbo --data data/cv_lb --split test --max-samples 300
!python evaluate.py --model openai/whisper-small --data data/cv_lb --split test --max-samples 300

## 3. Train
Smoke test first (a few minutes), then the real run. Re-run the same cell with `--resume` after a disconnect.

In [ ]:
!python train.py --config configs/turbo-lora.yaml --max-train-samples 300

In [ ]:
!python train.py --config configs/turbo-lora.yaml  # add --resume after a disconnect

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 4. Export, evaluate on the full test set, benchmark

In [ ]:
!python export.py --run runs/turbo-lora
!python evaluate.py --model runs/turbo-lora/merged --data data/cv_lb --split test

In [ ]:
!python benchmark.py --data data/cv_lb --max-samples 300 --models \
    openai/whisper-large-v3-turbo openai/whisper-small \
    unilux/whisper-medium-v1-luxembourgish ZLSCompLing/Whisper-Large-V3-Turbo-Luxembourgish \
    runs/turbo-lora/merged
!cat results/benchmark.md

## 5. Publish
Fill `model_card_template.md` with the numbers from `results/`, save it as `results/model_card.md`, then push.

In [ ]:
!python export.py --run runs/turbo-lora --card results/model_card.md \
    --push SitrakaForler/whisper-large-v3-turbo-lb --push-ct2 SitrakaForler/whisper-large-v3-turbo-lb-ct2